# Gemma 2 2B diagnostic

The observer signal in Gemma is weaker (+0.125) and peaks earlier (19% depth)
than GPT-2/Qwen (+0.28, 67% depth). This notebook runs four diagnostics:

1. **Full dense layer sweep** (all 26 layers) at 260 ex/dim — is there a hidden peak?
2. **Linear vs MLP probe** — is the signal there but nonlinear?
3. **Even vs odd layers** — does sliding window vs full attention matter?
4. **Activation geometry** — effective rank, norm, sparsity per layer

Run on Colab with A100 GPU.

In [ ]:
!git clone https://github.com/tmcarmichael/nn-observability.git
%cd nn-observability
!pip install uv -q
!uv sync --extra transformer -q

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_observe import collect_layer_data, load_wikitext, train_linear_binary, evaluate_head, bootstrap_ci, compute_hand_designed
from observe import compute_loss_residuals, partial_spearman

In [ ]:
# Load model in float16 for speed
model_id = 'google/gemma-2-2b'
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, torch_dtype=torch.float16).cuda()
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_layers = model.config.num_hidden_layers
hidden_dim = model.config.hidden_size
print(f'{n_layers} layers, {hidden_dim} dim')

# Load data
train_docs = load_wikitext('train', max_docs=2000)
test_docs = load_wikitext('test', max_docs=500)
print(f'{len(train_docs)} train docs, {len(test_docs)} test docs')

# 260 ex/dim token budget
max_train = 260 * hidden_dim  # ~599k
max_test = max_train // 2
print(f'Token budget: {max_train} train, {max_test} test ({max_train/hidden_dim:.0f} ex/dim)')

## 1. Full dense layer sweep (all 26 layers)

In [ ]:
layer_results = {}
for layer in range(n_layers):
    train_data = collect_layer_data(model, tokenizer, train_docs, layer, 'cuda', max_train)
    test_data = collect_layer_data(model, tokenizer, test_docs, layer, 'cuda', max_test)
    head = train_linear_binary(train_data, seed=42)
    _, rho, _ = evaluate_head(head, test_data)
    layer_results[layer] = float(rho)
    attn_type = 'sliding' if layer % 2 == 0 else 'full'
    print(f'  layer {layer:>2} ({attn_type:>7}): {rho:+.4f}')

peak = max(layer_results, key=layer_results.get)
print(f'\nPeak: layer {peak} ({peak/n_layers:.0%} depth) = {layer_results[peak]:+.4f}')

## 2. Linear vs MLP probe at peak layer

In [ ]:
# Collect at peak
train_data = collect_layer_data(model, tokenizer, train_docs, peak, 'cuda', max_train)
test_data = collect_layer_data(model, tokenizer, test_docs, peak, 'cuda', max_test)

# Linear probe (standard)
linear_rhos = []
for seed in [42, 43, 44]:
    head = train_linear_binary(train_data, seed=seed)
    _, rho, _ = evaluate_head(head, test_data)
    linear_rhos.append(rho)
print(f'Linear: {np.mean(linear_rhos):+.4f} +/- {np.std(linear_rhos):.4f}')

# MLP probe
mlp_rhos = []
for seed in [42, 43, 44]:
    torch.manual_seed(seed)
    np.random.seed(seed)
    acts = train_data['activations']
    residuals = compute_loss_residuals(train_data['losses'], train_data['max_softmax'], train_data['activation_norm'])
    targets = torch.from_numpy((residuals > 0).astype(np.float32))
    
    mlp = torch.nn.Sequential(
        torch.nn.Linear(hidden_dim, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1)
    )
    opt = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
    ds = torch.utils.data.TensorDataset(acts, targets)
    dl = torch.utils.data.DataLoader(ds, batch_size=1024, shuffle=True)
    mlp.train()
    for ep in range(20):
        for bx, by in dl:
            loss = F.binary_cross_entropy_with_logits(mlp(bx).squeeze(-1), by)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
    
    mlp.eval()
    with torch.inference_mode():
        scores = mlp(test_data['activations']).squeeze(-1).numpy()
    rho, _ = partial_spearman(scores, test_data['losses'], [test_data['max_softmax'], test_data['activation_norm']])
    mlp_rhos.append(rho)

print(f'MLP:    {np.mean(mlp_rhos):+.4f} +/- {np.std(mlp_rhos):.4f}')
print(f'MLP / Linear ratio: {np.mean(mlp_rhos) / np.mean(linear_rhos):.2f}')

## 3. Even (sliding window) vs odd (full attention) layers

In [ ]:
even_layers = [l for l in layer_results if l % 2 == 0]
odd_layers = [l for l in layer_results if l % 2 == 1]

even_mean = np.mean([layer_results[l] for l in even_layers])
odd_mean = np.mean([layer_results[l] for l in odd_layers])

print(f'Sliding window (even) layers mean: {even_mean:+.4f}')
print(f'Full attention (odd) layers mean:  {odd_mean:+.4f}')
print(f'Difference: {odd_mean - even_mean:+.4f}')

# Per-layer comparison
print('\nLayer-by-layer:')
for l in sorted(layer_results):
    attn = 'SW' if l % 2 == 0 else 'FA'
    bar = '+' * int(layer_results[l] * 100)
    print(f'  L{l:>2} [{attn}] {layer_results[l]:+.4f} {bar}')

## 4. Activation geometry per layer

In [ ]:
print(f'{"Layer":>5} {"Norm":>8} {"Sparsity":>10} {"Eff Rank":>10} {"Partial":>10}')
print('-' * 48)

for layer in range(0, n_layers, 2):  # every other layer for speed
    data = collect_layer_data(model, tokenizer, test_docs[:100], layer, 'cuda', 20000)
    acts = data['activations']
    
    norm = acts.norm(dim=1).mean().item()
    sparsity = (acts.abs() < 0.01).float().mean().item()
    
    # Effective rank via SVD on a sample
    sample = acts[:2000].numpy()
    s = np.linalg.svd(sample, compute_uv=False)
    p = s / s.sum()
    eff_rank = np.exp(-(p * np.log(p + 1e-10)).sum())
    
    pcorr = layer_results.get(layer, 0)
    print(f'{layer:>5} {norm:>8.1f} {sparsity:>10.3f} {eff_rank:>10.1f} {pcorr:>+10.4f}')

## Summary

In [ ]:
print('Gemma 2 2B diagnostic summary:')
print(f'  Peak layer: {peak} ({peak/n_layers:.0%} depth)')
print(f'  Linear peak partial corr: {layer_results[peak]:+.4f}')
print(f'  Linear 3-seed mean: {np.mean(linear_rhos):+.4f}')
print(f'  MLP 3-seed mean: {np.mean(mlp_rhos):+.4f}')
print(f'  MLP/Linear ratio: {np.mean(mlp_rhos)/np.mean(linear_rhos):.2f}')
print(f'  SW vs FA layer gap: {odd_mean - even_mean:+.4f}')
print()
if np.mean(mlp_rhos) > np.mean(linear_rhos) * 1.5:
    print('  --> Signal is substantially nonlinear. Linear probe undersells Gemma.')
elif np.mean(linear_rhos) > 0.20:
    print('  --> Signal is in the GPT-2/Qwen band with adequate data.')
elif np.mean(linear_rhos) > 0.10:
    print('  --> Signal exists but is genuinely weaker than GPT-2/Qwen.')
else:
    print('  --> Signal is marginal or absent.')

In [ ]:
# Save results
import json
results = {
    'model': model_id,
    'layer_profile': {str(k): v for k, v in sorted(layer_results.items())},
    'peak_layer': peak,
    'linear_rhos': linear_rhos,
    'mlp_rhos': [float(r) for r in mlp_rhos],
    'even_mean': float(even_mean),
    'odd_mean': float(odd_mean),
    'ex_per_dim': max_train / hidden_dim,
}
with open('gemma_diagnostic.json', 'w') as f:
    json.dump(results, f, indent=2)

from google.colab import files
files.download('gemma_diagnostic.json')